In [1]:
#assignment 2
#performance comparison of MLR and KNNR
#Dataset: EV Market 2026

In [2]:
#import required libraries

In [9]:
import pandas as pd
import numpy as np
import time 
import psutil
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline

In [10]:
# load the dataset

In [13]:
#read csv file using pandas
df = pd.read_csv(r"C:\Users\vkvar\Downloads\archive (1)\ev_market_2026.csv")
print("Dataset loaded successfully!")

Dataset loaded successfully!


In [14]:
# understand the dataset 

In [15]:

# Display all column names
print("\nColumn names:")
print(df.columns.tolist())


# Display information about the dataset
# This shows data types and number of non-empty values
print("\nDataset information:")
df.info()


# Check whether the dataset contains missing values
print("\nMissing values in each column:")
print(df.isnull().sum())



Column names:
['brand', 'model', 'year', 'variant', 'price_usd', 'battery_capacity_kwh', 'range_miles', 'charging_speed_kw', 'acceleration_0_60_mph', 'top_speed_mph', 'horsepower', 'torque_nm', 'drive_type', 'seating_capacity', 'body_type', 'cargo_volume_cubic_ft', 'weight_kg', 'safety_rating', 'autopilot_level', 'country_of_origin', 'market_segment', 'annual_sales_units', 'customer_rating', 'warranty_years']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   brand                  2000 non-null   str    
 1   model                  2000 non-null   str    
 2   year                   2000 non-null   int64  
 3   variant                2000 non-null   str    
 4   price_usd              2000 non-null   float64
 5   battery_capacity_kwh   2000 non-null   float64
 6   range_miles            2000 non-null   float64


In [16]:
# define features and target variables

In [17]:

# The target variable is the value that we want to predict
# In our dataset, we want to predict EV price
y = df["price_usd"]


# X contains the input features used to predict the price
# We remove price_usd because it is our target variable
X = df.drop(columns=["price_usd"])


# Display the number of input features
print("\nNumber of input features:", X.shape[1])


# Display the target variable
print("Target variable: price_usd")


Number of input features: 23
Target variable: price_usd


In [18]:
# identify numerical and categorical features

In [21]:
# Identify categorical features
categorical_features = X.select_dtypes(
    include=["object", "str"]
).columns.tolist()

# Identify numerical features
numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)


Categorical features:
['brand', 'model', 'variant', 'drive_type', 'body_type', 'country_of_origin', 'market_segment']

Numerical features:
['year', 'battery_capacity_kwh', 'range_miles', 'charging_speed_kw', 'acceleration_0_60_mph', 'top_speed_mph', 'horsepower', 'torque_nm', 'seating_capacity', 'cargo_volume_cubic_ft', 'weight_kg', 'safety_rating', 'autopilot_level', 'annual_sales_units', 'customer_rating', 'warranty_years']


In [22]:
#divide data into training and testing

In [23]:
# 80% of the data is used for training
# 20% of the data is used for testing
#
# random_state=42 makes the split reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


print("\nTraining data points:", len(X_train))
print("Testing data points:", len(X_test))



Training data points: 1600
Testing data points: 400


In [24]:
#preprocess data for MLR

In [31]:


# Numerical columns are kept as they are
#
# Categorical columns are converted into numerical
# values using One-Hot Encoding
#
# handle_unknown="ignore" prevents errors if a category
# appears in testing data but not in training data

preprocessor_mlr = ColumnTransformer(
    transformers=[
        (
            "numerical",
            "passthrough",
            numerical_features
        ),

        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [32]:
# create MLR model

In [34]:

# Create the MLR model
mlr_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor_mlr),

        ("regression", LinearRegression())
    ]
)



In [35]:
#measure MLR exceution

In [37]:
# Create a process object for measuring resources
process = psutil.Process()


# Record memory before running MLR
memory_before_mlr = process.memory_info().rss


# Record CPU time before running MLR
cpu_before_mlr = process.cpu_times()


# Record starting time
start_mlr = time.perf_counter()


# Train the MLR model
mlr_model.fit(X_train, y_train)


# Use the trained model to predict prices
mlr_prediction = mlr_model.predict(X_test)


# Record ending time
end_mlr = time.perf_counter()


# Record memory after running MLR
memory_after_mlr = process.memory_info().rss


# Record CPU time after running MLR
cpu_after_mlr = process.cpu_times()


In [38]:
#calculate MLR performance

In [40]:
# R² measures how well the model explains
# variation in the target variable
mlr_r2 = r2_score(
    y_test,
    mlr_prediction
)


# RMSE measures prediction error
# Lower RMSE generally means smaller prediction errors
mlr_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        mlr_prediction
    )
)


# MAE measures the average absolute prediction error
# Lower MAE means smaller average error
mlr_mae = mean_absolute_error(
    y_test,
    mlr_prediction
)


# Calculate execution time
mlr_time = end_mlr - start_mlr


# Calculate change in memory usage in MB
mlr_memory = (
    memory_after_mlr - memory_before_mlr
) / (1024 ** 2)


# Calculate CPU time used by the process
mlr_cpu_time = (
    (cpu_after_mlr.user - cpu_before_mlr.user)
    +
    (cpu_after_mlr.system - cpu_before_mlr.system)
)


In [41]:
#preprocess data for KNNR

In [48]:
# For KNN, numerical variables need to be standardized
# because KNN uses distances between observations
#
# StandardScaler changes numerical variables
# to a common scale

preprocessor_knn = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),

        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)


In [44]:
#create KNNR regression model

In [50]:
# n_neighbors=5 means the model considers
# the 5 nearest observations when making a prediction

knnr_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor_knn),
        (
            "regression",
            KNeighborsRegressor(
                n_neighbors=5
            )
        )
    ]
)

In [51]:
#MEASURE KNNR EXECUTION

In [53]:
# Record memory before running KNNR
memory_before_knn = process.memory_info().rss


# Record CPU time before running KNNR
cpu_before_knn = process.cpu_times()


# Record starting time
start_knn = time.perf_counter()


# Train the KNNR model
knnr_model.fit(X_train, y_train)


# Use the trained model to predict prices
knn_prediction = knnr_model.predict(X_test)


# Record ending time
end_knn = time.perf_counter()


# Record memory after running KNNR
memory_after_knn = process.memory_info().rss


# Record CPU time after running KNNR
cpu_after_knn = process.cpu_times()



In [54]:
#CALCULATE KNNR PERFORMANCE

In [56]:
# Calculate R²
knn_r2 = r2_score(
    y_test,
    knn_prediction
)


# Calculate RMSE
knn_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        knn_prediction
    )
)


# Calculate MAE
knn_mae = mean_absolute_error(
    y_test,
    knn_prediction
)


# Calculate execution time
knn_time = end_knn - start_knn


# Calculate memory change in MB
knn_memory = (
    memory_after_knn - memory_before_knn
) / (1024 ** 2)


# Calculate CPU time
knn_cpu_time = (
    (cpu_after_knn.user - cpu_before_knn.user)
    +
    (cpu_after_knn.system - cpu_before_knn.system)
)

In [57]:
#DISPLAY MLR RESULTS

In [59]:
print("\n")
print("=" * 60)
print("MULTIPLE LINEAR REGRESSION (MLR)")
print("=" * 60)

print("R²:", round(mlr_r2, 4))
print("RMSE:", round(mlr_rmse, 4))
print("MAE:", round(mlr_mae, 4))
print("Execution Time:", round(mlr_time, 6), "seconds")
print("CPU Time:", round(mlr_cpu_time, 6), "seconds")
print("Memory Change:", round(mlr_memory, 4), "MB")




MULTIPLE LINEAR REGRESSION (MLR)
R²: 0.8667
RMSE: 12328.7397
MAE: 8416.461
Execution Time: 0.091481 seconds
CPU Time: 0.09375 seconds
Memory Change: 0.3672 MB


In [60]:
#DISPLAY KNNR RESULTS

In [62]:
print("\n")
print("=" * 60)
print("K-NEAREST NEIGHBORS REGRESSION (KNNR)")
print("=" * 60)

print("R²:", round(knn_r2, 4))
print("RMSE:", round(knn_rmse, 4))
print("MAE:", round(knn_mae, 4))
print("Execution Time:", round(knn_time, 6), "seconds")
print("CPU Time:", round(knn_cpu_time, 6), "seconds")
print("Memory Change:", round(knn_memory, 4), "MB")




K-NEAREST NEIGHBORS REGRESSION (KNNR)
R²: 0.8437
RMSE: 13349.2662
MAE: 9159.2455
Execution Time: 0.293802 seconds
CPU Time: 0.28125 seconds
Memory Change: 0.375 MB


In [63]:
#CREATE FINAL COMPARISON TABLE

In [65]:
# Create a table containing the results
results = pd.DataFrame({

    "Metric": [
        "R²",
        "RMSE",
        "MAE",
        "Execution Time (seconds)",
        "CPU Time (seconds)",
        "Memory Change (MB)"
    ],

    "MLR": [
        mlr_r2,
        mlr_rmse,
        mlr_mae,
        mlr_time,
        mlr_cpu_time,
        mlr_memory
    ],

    "KNNR": [
        knn_r2,
        knn_rmse,
        knn_mae,
        knn_time,
        knn_cpu_time,
        knn_memory
    ]
})


# Display the final comparison table
print("\n")
print("=" * 60)
print("FINAL COMPARISON")
print("=" * 60)

print(
    results.to_string(index=False)
)



FINAL COMPARISON
                  Metric          MLR         KNNR
                      R²     0.866686     0.843702
                    RMSE 12328.739656 13349.266236
                     MAE  8416.461018  9159.245550
Execution Time (seconds)     0.091481     0.293802
      CPU Time (seconds)     0.093750     0.281250
      Memory Change (MB)     0.367188     0.375000


In [66]:
#SAVE RESULTS

In [67]:
# Save the comparison table as a CSV file
# This file can be uploaded to GitHub along with your code

results.to_csv(
    "MLR_vs_KNNR_results.csv",
    index=False
)


print("\nResults saved successfully!")
print("File created: MLR_vs_KNNR_results.csv")


Results saved successfully!
File created: MLR_vs_KNNR_results.csv
